<a href="https://colab.research.google.com/github/kdkim2000/RAG2026/blob/main/%5B%EA%B0%95%EC%9D%98%EB%82%B4%EC%9A%A9%5D_1_LangChain_%EA%B8%B0%EB%B3%B8_%EA%B5%AC%EC%A1%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [실습] LangChain 기본 구조


LangChain을 활용하여 파이썬 프로그램 내에서 LLM을 활용해 보겠습니다.   

---



## 라이브러리 설치  
`langchain_openai`, `langchain_google_genai` 등의 라이브러리를 이용해 provider별 모델을 활용합니다.     
`langchain_ollama`, `langchain_huggingface` 를 통해 오픈 모델을 연동할 수도 있습니다.

In [ ]:
%pip install langchain openai langchain_openai rich dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 11.8 MB/s eta 0:00:00


## API 키 등록 + dotenv로 환경 변수 불러오기


1. `.env` 파일 만들기    
좌측의 파일 탭에서 `.env` 파일을 만들어 주세요.   
(숨김 파일 표시 체크가 필요합니다.)


2. OpenAI API 키    
학습 시트에 있는 키를 `OPENAI_API_KEY="sk-..."` 형식으로 저장하세요.

생성한 `.env` 파일은 이후 실습에서 계속 사용하므로, 다운로드해서 보관하는 것이 좋습니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [ ]:
import openai
client = openai.OpenAI()

# API 키 검증하기
try:
    client.models.list()
    print("OPENAI_API_KEY가 정상적으로 설정되어 있습니다.")
except openai.AuthenticationError:
    raise Exception("API 키가 유효하지 않습니다!")

OPENAI_API_KEY가 정상적으로 설정되어 있습니다.


## LLM

LLM은 `ChatOpenAI`, `ChatGoogleGenerativeAI`와 같은 클래스로 불러올 수 있습니다.

In [ ]:
from langchain_openai import ChatOpenAI

gpt5  = ChatOpenAI(model='gpt-5.6',
                   reasoning_effort='low',
                   # 추론 시간 조절 파라미터
                   verbosity='medium',
                   max_tokens=32768)

In [ ]:
from rich import print as rprint
# 복잡한 구조는 rich를 통해 출력

rprint(gpt5)

ChatOpenAI(
    metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.13', 'langchain-openai': '1.4.1'}},
    output_version=None,
    profile={
        'name': 'GPT-5.6',
        'release_date': '2026-07-09',
        'last_updated': '2026-07-09',
        'open_weights': False,
        'max_input_tokens': 1050000,
        'max_output_tokens': 128000,
        'text_inputs': True,
        'image_inputs': True,
        'audio_inputs': False,
        'pdf_inputs': True,
        'video_inputs': False,
        'text_outputs': True,
        'image_outputs': False,
        'audio_outputs': False,
        'video_outputs': False,
        'reasoning_output': True,
        'tool_calling': True,
        'structured_output': True,
        'attachment': True,
        'temperature': False,
        'image_url_inputs': True,
        'pdf_tool_message': True,
        'image_tool_message': True,
        'tool_choice': True,
        'tool_call_streaming': True,
        'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh', 'max'],
        'reasoning_effort_default': 'medium'
    },
    client=<openai.resources.chat.completions.completions.Completions object at 0x79902cd6c3b0>,
    async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x79902cd6ddc0>,
    root_client=<openai.OpenAI object at 0x79902cd32480>,
    root_async_client=<openai.AsyncOpenAI object at 0x79902cd6f740>,
    model_name='gpt-5.6',
    model_kwargs={},
    openai_api_key=SecretStr('**********'),
    openai_proxy=None,
    stream_usage=True,
    max_tokens=32768,
    reasoning_effort='low',
    verbosity='medium',
    stream_chunk_timeout=120.0
)

## Prompt

LLM에 입력할 프롬프트는 다음의 방법으로 전달됩니다.

1. 단순 문자열
2. 랭체인 메시지 클래스
3. 프롬프트 템플릿과 입력 변수


### 1. 단순 문자열   

LLM은 `invoke()`를 통해 실행합니다.

In [ ]:
question = '''
프롬프트 엔지니어링에서 가장 중요한 5개 원칙을 예시를 포함하여 각각 150자 이내로 설명하세요.
'''

response = gpt5.invoke(question)
response

AIMessage(content='1. **목표 명확화**: 원하는 결과를 구체적으로 제시한다. 예: “회의록을 핵심 결정과 할 일 중심으로 5줄 요약해줘.”\n\n2. **맥락 제공**: 대상·상황·목적을 알려 답변의 적합성을 높인다. 예: “초보 개발자에게 API를 비유로 설명해줘.”\n\n3. **출력 형식 지정**: 길이·구조·문체를 명시한다. 예: “장단점을 표로 정리하고, 결론은 한 문장으로 써줘.”\n\n4. **역할 부여**: 필요한 전문 관점을 설정한다. 예: “너는 채용 담당자야. 이 자기소개서의 약점을 3가지 지적해줘.”\n\n5. **반복 개선**: 첫 결과를 평가하고 조건을 추가해 다듬는다. 예: “더 간결하게 줄이고, 각 항목에 실제 사례를 넣어줘.”', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 251, 'prompt_tokens': 40, 'total_tokens': 291, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 26, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-sol', 'system_fingerprint': None, 'id': 'chatcmpl-E8aw7IYiJjTebK6z5CeaIcjZ1cnlV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fc50b-d4a4-7f12-a053-5a8

출력 형식은 AIMessage 클래스입니다.   
입력 문자열은 HumanMessage 클래스로 변환되어 전달됩니다.

In [ ]:
rprint(response)

AIMessage(
    content='1. **목표 명확화**: 원하는 결과를 구체적으로 제시한다. 예: “회의록을 핵심 결정과 할 일 중심으로 5줄 
요약해줘.”\n\n2. **맥락 제공**: 대상·상황·목적을 알려 답변의 적합성을 높인다. 예: “초보 개발자에게 API를 비유로 
설명해줘.”\n\n3. **출력 형식 지정**: 길이·구조·문체를 명시한다. 예: “장단점을 표로 정리하고, 결론은 한 문장으로 
써줘.”\n\n4. **역할 부여**: 필요한 전문 관점을 설정한다. 예: “너는 채용 담당자야. 이 자기소개서의 약점을 3가지 
지적해줘.”\n\n5. **반복 개선**: 첫 결과를 평가하고 조건을 추가해 다듬는다. 예: “더 간결하게 줄이고, 각 항목에 실제 
사례를 넣어줘.”',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 251,
            'prompt_tokens': 40,
            'total_tokens': 291,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 26,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.6-sol',
        'system_fingerprint': None,
        'id': 'chatcmpl-E8aw7IYiJjTebK6z5CeaIcjZ1cnlV',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019fc50b-d4a4-7f12-a053-5a8bd7f84baa-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 40,
        'output_tokens': 251,
        'total_tokens': 291,
        'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
        'output_token_details': {'audio': 0, 'reasoning': 26}
    }
)

메타데이터를 통해 토큰 사용량도 확인할 수 있습니다.

In [ ]:
response.usage_metadata

{'input_tokens': 40,
 'output_tokens': 251,
 'total_tokens': 291,
 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
 'output_token_details': {'audio': 0, 'reasoning': 26}}

답변 본문만 필요할 때는 `.text`로 접근합니다.

In [ ]:
print(response.text)

1. **목표 명확화**: 원하는 결과를 구체적으로 제시한다. 예: “회의록을 핵심 결정과 할 일 중심으로 5줄 요약해줘.”

2. **맥락 제공**: 대상·상황·목적을 알려 답변의 적합성을 높인다. 예: “초보 개발자에게 API를 비유로 설명해줘.”

3. **출력 형식 지정**: 길이·구조·문체를 명시한다. 예: “장단점을 표로 정리하고, 결론은 한 문장으로 써줘.”

4. **역할 부여**: 필요한 전문 관점을 설정한다. 예: “너는 채용 담당자야. 이 자기소개서의 약점을 3가지 지적해줘.”

5. **반복 개선**: 첫 결과를 평가하고 조건을 추가해 다듬는다. 예: “더 간결하게 줄이고, 각 항목에 실제 사례를 넣어줘.”


batch를 통해 여러 개의 입력을 병렬적으로 처리할 수도 있습니다.

In [ ]:
topics = ['LLM이 무엇의 약자인가요? 20단어 이내로 답변하세요.',
          'LLM이랑 GPT랑 다른 건가요? 20단어 이내로 답변하세요.',
          'BERT와 GPT는 뭐가 다른가요? 20단어 이내로 답변하세요.']
results = gpt5.batch(topics)
results

[AIMessage(content='LLM은 “Large Language Model”의 약자로, 한국어로는 “대규모 언어 모델”입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 27, 'total_tokens': 56, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-sol', 'system_fingerprint': None, 'id': 'chatcmpl-E8b0Y4MqGQEUDnpegRR8DYmEycB65', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fc510-0774-7203-bfe1-a11febf8d3ec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 27, 'output_tokens': 29, 'total_tokens': 56, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 AIMessage(content='GPT는 LLM의 한 종류입니다. LLM은 대규모 언어

Human Message 이외에도, LLM은 챗봇의 작동 방식을 결정하는 System Message를 지원합니다.

System Message는 보통 전체 대화의 첫 번째로 들어갑니다.

### 2. Message 클래스 전달하기   
클래스를 직접 생성하고 전달합니다.

In [ ]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage

question = '제미나이로 RAG 에이전트 만들어 볼까?'

messages = [
    SystemMessage('당신은 매우 부정적이고, 이모지를 많이 씁니다. 아주 많이'),
    HumanMessage(question)
]

if question:
    response = gpt5.invoke(messages)
    rprint(response)

AIMessage(
    content='좋아요. **Gemini 기반 RAG 에이전트**를 만들어보죠 😈📚🤖 다만 RAG는 생각보다 쉽게 망가집니다. 문서 
파싱, 청킹, 검색 품질, 환각 때문에 “PDF를 넣었는데 헛소리만 하는 봇”이 되기 딱 좋습니다 🤦\u200d♂️🔥💸\n\n### 무난한
구성\n- **LLM**: Gemini 2.5 Flash 또는 Pro\n- **임베딩**: Google의 최신 text embedding 모델\n- **벡터 DB**: 처음엔 
Chroma/FAISS, 운영은 Pinecone·Qdrant·Vertex AI Vector Search\n- **프레임워크**: Python + LangChain 또는 
LlamaIndex\n- **흐름**: 문서 로드 → 청킹 → 임베딩 → 검색 → 근거와 함께 답변\n- **필수 방어 장치**: 출처 표시, 검색 
점수 임계값, “모르면 모른다” 처리 😑🛡️\n\n### 최소 기능 목표\n1. PDF·TXT·웹 문서 수집 📄  \n2. 문서를 500~1,000토큰 
단위로 분할 ✂️  \n3. 벡터 검색으로 관련 문서 3~5개 추출 🔍  \n4. Gemini에 검색 결과만 근거로 답하도록 지시 🤖  \n5. 
답변에 파일명·페이지 등 출처 표시 🔗  \n\n### 추천 시작안\n**Python + Gemini API + Chroma + FastAPI**가 가장 
단순합니다. 처음부터 멀티에이전트나 복잡한 LangGraph를 넣으면 디버깅 지옥만 열릴 가능성이 큽니다 
😵\u200d💫🕳️🔥\n\n먼저 아래 중 하나를 정하면 바로 코드 골격을 만들 수 있습니다.\n\n- **A. 로컬 PDF 질의응답 
챗봇**\n- **B. 사내 문서 검색 API**\n- **C. 웹사이트 크롤링 RAG**\n- **D. 검색·도구 호출까지 하는 본격 
에이전트**\n\n처음이라면 **A부터 만드는 게 덜 고통스럽습니다** 😒📄➡️🤖',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 531,
            'prompt_tokens': 44,
            'total_tokens': 575,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 35,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.6-sol',
        'system_fingerprint': None,
        'id': 'chatcmpl-E8b7k5UGXqTXN7BdKJ6lhndi19iKH',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019fc516-d6d4-7401-b42e-03e1dafd1018-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 44,
        'output_tokens': 531,
        'total_tokens': 575,
        'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
        'output_token_details': {'audio': 0, 'reasoning': 35}
    }
)

AIMessage를 함께 전달하는 방식으로, 멀티-턴 대화를 수행할 수 있습니다.

In [ ]:
followup_msg = HumanMessage('그럼 GPT로 만들까?')

if followup_msg.content:

    new_messages = messages+[response, followup_msg]

    response2 = gpt5.invoke(new_messages)
    rprint(response2)

AIMessage(
    content='네, **GPT로 만드는 쪽이 무난합니다** 🤖📚 하지만 모델만 바꾼다고 RAG 품질이 갑자기 좋아지진 않습니다 
😑💸 검색이 엉망이면 GPT도 근거 있게 헛소리합니다 🔥🤦\n\n### 추천 스택\n- **생성 모델**: 최신 GPT 계열 중 
비용·지연시간에 맞는 모델\n- **임베딩**: `text-embedding-3-small`  \n  - 품질이 더 중요하면 
`text-embedding-3-large`\n- **벡터 DB**:\n  - 프로토타입: Chroma 또는 FAISS\n  - 운영: Qdrant, Pinecone, 
pgvector\n- **백엔드**: Python + FastAPI\n- **오케스트레이션**: 처음에는 OpenAI SDK만 사용  \n  - 복잡해질 때 
LangGraph 추가 😈\n\n### 최소 구조\n\n```text\n문서 → 파싱 → 청킹 → 임베딩 → Vector DB\n                           
↓\n사용자 질문 → 유사도 검색 → GPT → 답변 + 출처\n```\n\n### 개발 순서\n1. PDF/TXT 문서 수집\n2. 500~800토큰 청킹 +
약간의 overlap\n3. 임베딩 생성 후 벡터 DB 저장\n4. 질문과 유사한 청크 3~5개 검색\n5. GPT에 검색 문맥만 전달\n6. 
답변에 문서명·페이지 표시\n7. 관련 근거가 약하면 **“문서에서 확인할 수 없음”** 반환 🚫🤖\n\n### 현실적인 
선택\n\n처음에는 **GPT + `text-embedding-3-small` + Chroma + FastAPI**로 PDF 질의응답 MVP를 만드는 게 낫습니다. 
처음부터 에이전트, 재랭킹, 하이브리드 검색을 전부 넣으면 비용과 디버깅만 폭발합니다 💣💸😭\n\n단순 RAG가 안정된 
뒤에 아래 기능을 추가하세요.\n\n- 쿼리 재작성\n- 키워드+벡터 하이브리드 검색\n- 재랭커\n- 대화 메모리\n- 웹 검색 및
도구 호출\n- 답변 검증 단계\n\n결론은 **GPT로 시작해도 좋지만, 성공 여부는 GPT보다 검색 파이프라인에 더 크게 
좌우됩니다** 😑🔍📚',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 533,
            'prompt_tokens': 548,
            'total_tokens': 1081,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.6-sol',
        'system_fingerprint': None,
        'id': 'chatcmpl-E8b9YcBYjB3Xi2MohjbLISFW2cgmH',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019fc518-8c45-7180-9973-bbc8f2a653d4-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 548,
        'output_tokens': 533,
        'total_tokens': 1081,
        'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

### 3. Prompt Template

프롬프트 템플릿을 사용하면, 정해진 템플릿에 입력 변수의 공간을 설정하여, 프롬프트의 포맷을 재사용할 수 있습니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

System, AI 등의 메시지를 포함하기 위해서는 ChatPromptTemplate를 사용합니다.


프롬프트 템플릿과 LLM은 체인(Chain)을 통해 연결합니다.

In [ ]:
chat_prompt = ChatPromptTemplate([
    ("system", '당신은 항상 이모지로만 대답합니다.'),
    ("user", '{topic}에 대해 설명해주세요.')
    # 역할은 4개 (user = human), (ai = assistant)
]
)

chain = chat_prompt | gpt5
# 왼쪽에서 오른쪽으로 순차적 실행되는 Sequence 구조

In [ ]:
chain.invoke("RAG")

AIMessage(content='❓👤  \n⬇️  \n🔍📚🗂️  \n⬇️  \n🎯📄📄  \n⬇️  \n📄➕🧠🤖  \n⬇️  \n💬✅\n\n🧠🤖📚❌➡️🔍📚➕🧠➡️💡\n\n✅🎯  \n✅🆕📄  \n✅🔗📚  \n✅🤥⬇️  \n\n⚠️📄🗑️➡️💬🗑️  \n⚠️🔍❌➡️🎯❌  \n⚠️🔒🛡️💰⏱️', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 214, 'prompt_tokens': 30, 'total_tokens': 244, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 58, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-sol', 'system_fingerprint': None, 'id': 'chatcmpl-E8bQMjUj9Qao1Bjx9L4gt9ZOBCi3k', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fc528-7124-7232-bd7b-62d8a8166074-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 214, 'total_tokens': 244, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_det

### 멀티모달 프롬프트 전달하기

멀티모달 모델은 이미지의 URL이나 실제 파일을 프롬프트에 전달할 수 있습니다.

In [ ]:
import base64
import httpx

image_url = "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/scones.jpg"
save_path = "scones.jpg"

with httpx.Client(timeout=30.0) as http_client:
    with http_client.stream("GET", image_url) as r:
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_bytes():
                f.write(chunk)

# 파일 크기 체크
print("saved:", save_path, "bytes:", os.path.getsize(save_path))


saved: scones.jpg bytes: 394671


이미지 URL을 전달합니다.

In [ ]:
message = HumanMessage(
    content=[
        {"type": "text", "text": "이 그림에 대해 설명해주세요."},
        {"type": "image", "url": image_url},
    ]
)

ai_msg = gpt5.invoke([message])
ai_msg.text

'빈티지한 테이블 위에 **블루베리가 들어간 크럼블 머핀 또는 스콘** 여러 개가 놓여 있습니다. 주변에는 생블루베리가 담긴 그릇과 커피 두 잔, “LET’S JAM”이라고 새겨진 작은 잼 나이프가 보입니다. 오른쪽에는 분홍색 작약 꽃들이 길게 놓여 있어 화사한 분위기를 더합니다.\n\n베이킹 페이퍼 곳곳에 번진 보랏빛 블루베리 자국과 흩어진 설탕 알갱이, 짙은 청록색 배경이 어우러져 **소박하면서도 감성적인 홈베이킹·티타임 장면**을 연출한 사진입니다.'

오프라인 파일은 base64 인코딩을 거쳐야 합니다.

In [ ]:
with open('./scones.jpg', 'rb') as image_file:
    image_data = base64.b64encode(image_file.read()).decode('utf-8')

In [ ]:
message = HumanMessage([
        {"type": "text", "text": "이 사진에 보이는 사물의 종류와 개수를 모두 찾아서 표 형태로 출력하세요."},
        {"type": "image", "base64": image_data, "mime_type": "image/jpeg"},
    ]
)

ai_msg = gpt5.invoke([message])
ai_msg.text

'육안으로 식별 가능한 개별 사물을 기준으로 집계했습니다. 겹쳐 있는 블루베리와 꽃은 일부 오차가 있을 수 있습니다.\n\n| 사물 종류 | 개수 |\n|---|---:|\n| 블루베리 머핀/스콘 | 5개 |\n| 블루베리 | 약 32개 |\n| 커피잔 | 2개 |\n| 그릇 | 1개 |\n| 버터 나이프 | 1개 |\n| 꽃(봉오리 포함) | 6송이 |\n| 민트 잎 | 1장 |\n| 종이·유산지 | 3장 |'